# Speed Prediction — 파생/추가 피처 빠른 테스트 (LightGBM)

**목적**: `11_speed_model_baseline.ipynb`의 LightGBM 결과를 기준점으로 두고, 파생 피처와 신규 외부 피처를 추가했을 때 실제로 개선되는지 확인합니다.

**핵심 전제**: 여기서 만드는 피처들은 전부 **미관측 링크(29.25%)에도 쓸 수 있는 범용 피처** 후보를 찾는 작업입니다 — 관측 링크는 자기 자신의 과거 속도 이력(Model A 방식)을 쓰면 되므로 이 노트북의 범위가 아닙니다.

**평가 방식 변경(이번)**: 지금까지 매 버전마다 무작위 분할로 먼저 보고 마지막에 GroupKFold로 검산하는 2단계였는데, **무작위 분할을 뺐습니다.** 반복해서 확인한 결과 무작위 분할은 매번 GroupKFold보다 R²가 0.07~0.09 높게 나왔습니다(6차 랜덤 0.746 vs GroupKFold 0.680, 7차 랜덤 0.780 vs GroupKFold 0.708) — 같은 링크의 다른 시간대 행이 대부분 train에 있어서 이웃속도 피처가 거의 항상(93~97%) 살아있는, 우리가 실제로 풀어야 하는 "한 번도 안 본 도로" 문제보다 훨씬 쉬운 문제였기 때문입니다. `src/11` 공식 GroupKFold(0.562)가 노트북11의 무작위분할(0.610)보다 낮았던 첫 사례부터 이미 같은 신호였습니다. 이제부터는 **`LINK_ID` 기준 GroupKFold(공간 일반화) 하나만** 정식 평가로 씁니다 — 계산량도 절반 이상 줄어듭니다.

**버전 변경 이력**
- 1~4차, src/11 공식 평가: `data/README.md`·이전 커밋 참고(무작위분할 시절 기록)
- **5~7차**: 데이터셋 전환(dimension+fact) → 생활인구 추가 → 도로망 위상·강수파생 추가. GroupKFold 기준 R²: 0.680(7차, 단일모델).
- **8차(B/C 분리모델)**: `notebooks/13_tier_split_evaluation.ipynb`와 `src/routing/13_predict_all_link_speed.py`가 실전 적용한 방식. 이웃속도 피처가 있으면(B 계층) 도움되지만 없으면(C 계층) 해가 되는 문제를, 모델을 둘로 나눠(이웃속도 포함 모델 / 정적 전용 모델) 해결합니다.
- **9차(이번, 신규 외부 데이터)**: `AverageSpeed(LINK).xlsx`(2024년 링크별 실측 평균속도, 연간+시간대별), `CongestIndex(LINK).xlsx`(혼잡빈도·시간·기대강도)를 추가합니다. 사전 확인 결과 둘 다 Level5.5(주요도로망) 기준이라 `traffic_volume_aadt`와 **정확히 같은 17,929개 링크**만 커버합니다(C 계층 커버리지 1.1%로 동일 — 진짜 필요한 Level6 상세도로망 데이터는 공문이 필요해 아직 확보 못함). 다만 `annual_avg_speed`는 우리 실측 속도와 상관계수 0.787로 지금까지 중 가장 강한 단일 피처 후보이고, `MAX_SPD`와의 상관은 0.460뿐이라 완전 중복은 아닙니다 — 커버되는 곳에서 실제로 도움되는지 확인합니다.

**캐싱**: 원본 결합 결과를 `data/interim/notebook_cache/nb12_feature_cache_v7.parquet`에 캐싱합니다(9차 신규 피처로 스키마가 또 바뀌어 v6과 분리).

In [1]:
import gc
from pathlib import Path

import geopandas as gpd
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DOCS_FIGURES_DIR = PROJECT_ROOT / "docs" / "figures" / "model"
DOCS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# 새 스키마: 정적(링크당 1행) + 동적(링크x날짜x시간) 분리, LINK_ID로 merge
DIMENSION_PATH = PROCESSED_DIR / "seoul_link_dimension.csv"
FACT_PATH = PROCESSED_DIR / "seoul_link_speed_fact.csv"
SEOUL_LINK_SHP = INTERIM_DIR / "seoul_link.shp"

# 생활인구(실험용, 아직 파이프라인에 안 붙음) — notebooks/14_accident_hotspot_classification.ipynb와 동일 원본
GRID_MATCH_SHP = RAW_DIR / "traffic" / "match" / "match.shp"                 # 250m격자 CELL_ID -> geometry
GRID_POP_DIR = RAW_DIR / "traffic" / "250_LOCAL_RESD_202607"                 # 250m 격자 생활인구(일별)
DONG_POP_DIR = RAW_DIR / "traffic" / "250_LOCAL_RESD_ADMDONG_202607"         # 행정동 생활인구(일별)

# 9차(신규): View-T Level5.5 실측 평균속도·혼잡지표 — traffic_volume과 동일 매칭방식(ITS_LINK_ID 콤마 explode)
AVG_SPEED_XLSX = RAW_DIR / "traffic" / "AverageSpeed(LINK).xlsx"
CONGEST_INDEX_XLSX = RAW_DIR / "traffic" / "CongestIndex(LINK).xlsx"

CACHE_PATH = INTERIM_DIR / "notebook_cache" / "nb12_feature_cache_v7.parquet"
CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

RANDOM_STATE = 42
COLOR_RMSE = "#2a78d6"   # dataviz 팔레트 slot 1 blue
COLOR_MAE = "#eb6834"    # dataviz 팔레트 slot 2 orange
COLOR_SEQ = "#3987e5"

## 1. 데이터 로드 (dimension + fact 결합, + 캐싱)

`seoul_link_speed_fact.csv`(동적, 15,094,335행)에 `seoul_link_dimension.csv`(정적, 63,548행)를 `LINK_ID`로 결합합니다. `bus_lane_ratio`/`school_zone_count`/`accident_hotspot_count`/`speed_bump_count`/`traffic_volume_aadt`는 이미 dimension에, `traffic_volume_hourly`는 이미 fact에 들어있어(둘 다 `src/pipeline/04`·`05`가 미리 계산) 구 1~3절의 공간조인이 필요 없습니다.

**신규(7차)**: 지금까지 dimension에 있는데도 안 쓰던 **도로망 위상·파생 28개**(`TOPO_COLS`)를 여기서 같이 가져옵니다 — `notebooks/13_tier_split_evaluation.ipynb`가 검증한 목록과 동일합니다.

In [2]:
NUMERIC_COLS = [
    "LANES", "MAX_SPD", "LENGTH", "시간코드",
    "elev_start_m", "elev_mid_m", "elev_end_m", "elev_mean_m", "elev_min_m", "elev_max_m", "elev_delta_m",
    "slope_pct", "abs_slope_pct", "precip_mm",
]
CATEGORICAL_COLS = [
    "ROAD_RANK", "ROAD_TYPE", "ROAD_USE", "MULTI_LINK", "CONNECT",
    "REST_VEH", "REST_W", "REST_H", "C-ITS",
]
# pipeline/04가 이미 0으로 채워둔 외부 컨텍스트(결측 없음)
EXTERNAL_CONTEXT_COLS = ["bus_lane_ratio", "school_zone_count", "accident_hotspot_count", "speed_bump_count"]
# 결측이 정상인 컬럼(전체 매칭 안 됨) — NUMERIC_COLS의 dropna 대상에서 제외하고 LightGBM 기본 결측치 처리에 맡김
SPARSE_NUMERIC_COLS = ["traffic_volume_aadt", "traffic_volume_hourly"]
# 도로망 위상·파생 28개(신규, 7차) — pipeline/04 산출, notebooks/13_tier_split_evaluation.ipynb 검증 목록과 동일
TOPO_COLS = [
    "f_node_node_in_degree", "f_node_incoming_max_spd_mean", "f_node_incoming_lanes_mean",
    "f_node_node_out_degree", "f_node_outgoing_max_spd_mean", "f_node_outgoing_lanes_mean",
    "f_node_node_total_degree",
    "t_node_node_in_degree", "t_node_incoming_max_spd_mean", "t_node_incoming_lanes_mean",
    "t_node_node_out_degree", "t_node_outgoing_max_spd_mean", "t_node_outgoing_lanes_mean",
    "t_node_node_total_degree",
    "upstream_max_spd_mean", "downstream_max_spd_mean", "upstream_lanes_mean", "downstream_lanes_mean",
    "node_complexity", "downstream_speed_delta", "slope_pct_clipped", "slope_outlier_flag",
    "is_uphill", "is_downhill", "steep_uphill", "slope_length_m", "max_spd_x_lanes",
    "length_x_node_complexity",
]
TARGET = "평균속도"

if CACHE_PATH.exists():
    df = pd.read_parquet(CACHE_PATH)
    print("캐시에서 로드:", CACHE_PATH)
else:
    dim_dtype = {c: str for c in CATEGORICAL_COLS}
    dim_dtype["LINK_ID"] = str
    dimension = pd.read_csv(DIMENSION_PATH, dtype=dim_dtype, encoding="utf-8-sig")
    dim_cols = (["LINK_ID"] + CATEGORICAL_COLS
                + ["LANES", "MAX_SPD", "LENGTH", "elev_start_m", "elev_mid_m", "elev_end_m",
                   "elev_mean_m", "elev_min_m", "elev_max_m", "elev_delta_m", "slope_pct", "abs_slope_pct"]
                + EXTERNAL_CONTEXT_COLS + ["traffic_volume_aadt"] + TOPO_COLS)

    fact = pd.read_csv(
        FACT_PATH, dtype={"LINK_ID": str, "기준일": str}, encoding="utf-8-sig",
        usecols=["LINK_ID", "기준일", "시간코드", TARGET, "precip_mm", "traffic_volume_hourly"],
    )

    df = fact.merge(dimension[dim_cols], on="LINK_ID", how="left", validate="many_to_one")

    for col in NUMERIC_COLS + [TARGET]:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")
    for col in EXTERNAL_CONTEXT_COLS:
        df[col] = df[col].astype("float32")
    for col in SPARSE_NUMERIC_COLS:
        df[col] = df[col].astype("float32")
    for col in TOPO_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

    RENAME_MAP = {"C-ITS": "c_its"}
    df = df.rename(columns=RENAME_MAP)
    CATEGORICAL_COLS = [RENAME_MAP.get(c, c) for c in CATEGORICAL_COLS]

    df = df.dropna(subset=[TARGET] + NUMERIC_COLS)
    df.to_parquet(CACHE_PATH)
    print("새로 만들어서 캐시 저장:", CACHE_PATH)

CATEGORICAL_COLS = [c if c != "C-ITS" else "c_its" for c in CATEGORICAL_COLS]
for col in CATEGORICAL_COLS:
    df[col] = df[col].astype("category")

print("행 수:", len(df), "| 컬럼 수:", df.shape[1])
print("메모리 사용량(MB):", round(df.memory_usage(deep=True).sum() / 1e6, 1))
print()
print("traffic_volume_hourly 결측 비율:", round(df["traffic_volume_hourly"].isna().mean() * 100, 1), "%")
print("traffic_volume_aadt 결측 비율:", round(df["traffic_volume_aadt"].isna().mean() * 100, 1), "%")
print("도로망 위상·파생 결측 비율(최대):", round(df[TOPO_COLS].isna().mean().max() * 100, 1), "%")
df.head()

캐시에서 로드: C:\Users\ejjun\Desktop\seoul-eco-routing\data\interim\notebook_cache\nb12_feature_cache_v7.parquet


행 수: 15094335 | 컬럼 수: 60
메모리 사용량(MB): 3607.5

traffic_volume_hourly 결측 비율: 60.7 %
traffic_volume_aadt 결측 비율: 60.7 %


도로망 위상·파생 결측 비율(최대): 0.0 %


,LINK_ID,기준일,시간코드,평균속도,precip_mm,traffic_volume_hourly,ROAD_RANK,ROAD_TYPE,ROAD_USE,MULTI_LINK,...,node_complexity,downstream_speed_delta,slope_pct_clipped,slope_outlier_flag,is_uphill,is_downhill,steep_uphill,slope_length_m,max_spd_x_lanes,length_x_node_complexity
0,1000000301,20260707,1.0,13.166667,0.0,398.0,103,0,0,1,...,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,275.356628
1,1000000301,20260707,2.0,16.250000,0.0,317.0,103,0,0,1,...,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,275.356628
2,1000000301,20260707,3.0,14.000000,0.0,213.0,103,0,0,1,...,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,275.356628
3,1000000301,20260707,4.0,15.000000,0.0,180.0,103,0,0,1,...,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,275.356628
4,1000000301,20260707,5.0,15.250000,0.0,255.0,103,0,0,1,...,6.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,275.356628


## 2. 파생 피처 생성 (요일/주말/시간 순환 인코딩 + 강수·첨두시)

캐시 이후 단계라 로직을 바꿔도 캐시 무효화가 필요 없습니다.

**신규(7차)**: `src/11_evaluate_speed_model.py`가 쓰는 강수·첨두시 파생 피처를 추가합니다 — `log_precip`, `rain_class`, `is_morning_peak`/`is_evening_peak`, `rain_morning_peak`/`rain_evening_peak`, `rain_x_max_spd`. 첨두시 판정은 **실제 시각**(`시간코드 - 1`) 기준으로 합니다 — `src/11`은 `시간코드`를 보정 없이 그대로 써서 실제로는 한 시간 이른 구간(예: 06~08시)을 "오전 첨두"로 잡고 있었습니다.

In [3]:
df["_date"] = pd.to_datetime(df["기준일"], format="%Y%m%d")
df["dow"] = df["_date"].dt.weekday.astype("category")
df["is_weekend"] = (df["_date"].dt.weekday >= 5).astype("int8")

hour_code = df["시간코드"].astype("float32")
df["hour_sin"] = np.sin(2 * np.pi * hour_code / 24).astype("float32")
df["hour_cos"] = np.cos(2 * np.pi * hour_code / 24).astype("float32")

# 실제 시각 = 시간코드 - 1 (00시대=시간코드01 규칙, pipeline/02 근거) — 첨두시 판정은 이 기준으로
actual_hour = (df["시간코드"].astype("int16") - 1)
df["is_morning_peak"] = actual_hour.isin([7, 8, 9]).astype("int8")
df["is_evening_peak"] = actual_hour.isin([17, 18, 19]).astype("int8")

precip = df["precip_mm"].clip(lower=0)
df["is_rain"] = (precip > 0).astype("int8")
df["log_precip"] = np.log1p(precip).astype("float32")
df["rain_class"] = pd.cut(
    precip, bins=[-np.inf, 0, 1, 5, 10, np.inf],
    labels=["none", "light", "moderate", "heavy", "very_heavy"],
).astype(str).astype("category")
df["rain_morning_peak"] = (df["is_rain"] * df["is_morning_peak"]).astype("int8")
df["rain_evening_peak"] = (df["is_rain"] * df["is_evening_peak"]).astype("int8")
df["rain_x_max_spd"] = (df["log_precip"] * df["MAX_SPD"]).astype("float32")

df = df.drop(columns=["_date"])


## 3. 인접 실측 이웃 속도

**목적**: 지금 만드는 피처들은 전부 "미관측 링크에도 쓸 수 있는 범용 피처" 후보를 찾는 작업입니다(관측 링크 자체는 나중에 Model A처럼 자기 이력을 쓰면 되니 여기서는 다루지 않습니다). 이번엔 "이 링크와 도로망으로 맞닿은 링크가 **그 시각에 실제로 관측된 값**"을 피처로 추가합니다(이웃의 예측값이 아니라 실측값 — 순환 의존 문제 없음).

사전 검증(별도 확인): 관측 링크의 97.0%가 관측된 1홉 이웃을 갖고 있고, 자기 실제속도와 이웃의 실제속도 상관계수는 0.685였습니다. 미관측 링크 기준으로는 75.3%가 1홉 안에 실측 이웃을 갖습니다.

**계산 방식**: `F_NODE`/`T_NODE` 위상으로 1홉 인접 그래프를 만들고, 링크 x 시간대(LINK_ID x tm) 속도 행렬에 희소 인접행렬을 곱해 "1홉 관측 이웃들의 그 시각 평균 실제속도"를 한 번에 계산합니다(결측 이웃은 자동으로 평균에서 제외). 1홉 안에 관측 이웃이 없는 링크는 NaN으로 남겨 LightGBM 기본 결측 처리에 맡깁니다.

**주의**: 여기서 만드는 `nearest_obs_neighbor_speed`는 전체 데이터 기준 계산이라 그대로 평가에 쓰면 안 됩니다(같은 링크가 train/test 양쪽에 실측값을 흘려보내 누수됩니다) — 7절 GroupKFold에서 폴드마다 train 링크만으로 다시 계산합니다(`build_neighbor_feature_for_allowed`). 여기서 만든 버전은 이후 파생 피처 계산과 데이터 탐색용입니다.

In [4]:
import geopandas as gpd
from collections import defaultdict
from scipy.sparse import csr_matrix

seoul_link_topo = gpd.read_file(SEOUL_LINK_SHP)[["LINK_ID", "F_NODE", "T_NODE"]]
seoul_link_topo["LINK_ID"] = seoul_link_topo["LINK_ID"].astype(str)

node_links = defaultdict(set)
for r in seoul_link_topo.itertuples(index=False):
    node_links[r.F_NODE].add(r.LINK_ID)
    node_links[r.T_NODE].add(r.LINK_ID)
link_neighbors = defaultdict(set)
for node, links in node_links.items():
    for l in links:
        link_neighbors[l].update(links - {l})

df["tm_key"] = df["기준일"].astype(str) + "_" + df["시간코드"].astype(int).astype(str)

link_ids_sorted = np.sort(df["LINK_ID"].unique())
link_idx = {l: i for i, l in enumerate(link_ids_sorted)}
observed_set = set(link_ids_sorted)
n_links = len(link_ids_sorted)

rows_idx, cols_idx = [], []
for l in link_ids_sorted:
    i = link_idx[l]
    for n in link_neighbors.get(l, ()):
        if n in observed_set:
            rows_idx.append(i)
            cols_idx.append(link_idx[n])
A = csr_matrix((np.ones(len(rows_idx), dtype="float32"), (rows_idx, cols_idx)), shape=(n_links, n_links))
print("인접(관측-관측) 엣지 수:", len(rows_idx), "| 1홉 관측 이웃이 0개인 링크:",
      (np.asarray(A.sum(axis=1)).ravel() == 0).sum())

speed_wide = df.pivot_table(index="LINK_ID", columns="tm_key", values=TARGET, aggfunc="mean")
speed_wide = speed_wide.reindex(link_ids_sorted)

valid = (~speed_wide.isna()).to_numpy(dtype="float32")
filled = np.nan_to_num(speed_wide.to_numpy(dtype="float32"), nan=0.0)

numerator = A @ filled
denom = A @ valid
with np.errstate(invalid="ignore", divide="ignore"):
    neighbor_mean = numerator / denom
neighbor_mean[denom == 0] = np.nan

tm_columns = list(speed_wide.columns)  # 10번 섹션(폴드별 재계산)에서 재사용

neighbor_long = pd.DataFrame(neighbor_mean, index=speed_wide.index, columns=speed_wide.columns) \
    .reset_index().melt(id_vars="LINK_ID", var_name="tm_key", value_name="nearest_obs_neighbor_speed")
df = df.merge(neighbor_long, on=["LINK_ID", "tm_key"], how="left")
df = df.drop(columns=["tm_key"])

# filled/valid(링크x시간 실측 행렬)와 link_neighbors/link_idx는 10번 섹션(GroupKFold, 폴드별 이웃
# 재계산)에서 다시 쓰므로 지우지 않음 — speed_wide/A(전체 기준)와 중간 계산값만 정리
del speed_wide, numerator, denom, neighbor_mean, neighbor_long
gc.collect()

print("nearest_obs_neighbor_speed 결측 비율:", round(df["nearest_obs_neighbor_speed"].isna().mean() * 100, 1), "%")
print(df["nearest_obs_neighbor_speed"].describe())


인접(관측-관측) 엣지 수: 145764 | 1홉 관측 이웃이 0개인 링크: 1334


nearest_obs_neighbor_speed 결측 비율: 3.0 %


count    1.464761e+07
mean     2.300422e+01
std      1.039420e+01
min      1.000000e+00
25%      1.662500e+01
50%      2.064583e+01
75%      2.613750e+01
max      1.079375e+02
Name: nearest_obs_neighbor_speed, dtype: float64


## 4. (신규) 생활인구 — 격자/행정동

`notebooks/14_accident_hotspot_classification.ipynb`(다른 브랜치, 사고다발지역 분류)에서 처음 쓴 서울시 생활인구 데이터를 속도 예측에도 추가합니다. 아직 `pipeline/04`/`05`나 `seoul_link_dimension.csv`에는 안 붙어 있는 **실험 단계** 데이터입니다.

- **격자 생활인구**(`250_LOCAL_RESD_202607/`): 250m 격자(`CELL_ID`) × 일자 × 시간 단위. 도로 링크 스케일에 가장 가까운 해상도.
- **행정동 생활인구**(`250_LOCAL_RESD_ADMDONG_202607/`): 행정동(8자리 코드) × 일자 × 시간 단위. 훨씬 넓은 면적.
- 링크 → 격자 매칭은 `match/match.shp`(`CELL_ID`+geometry, EPSG:5179)에 링크 중심점이 속하는 셀을 찾는 방식(`sjoin(predicate="within")`).
- 링크 → 행정동은 별도 경계 파일 없이, **격자 생활인구 파일 자체의 `행정동코드` 컬럼**을 이용합니다 — 경계에 걸친 격자는 인구가 여러 행정동에 나뉘어 기록되므로, 셀별로 인구 비중이 가장 큰 행정동을 그 셀의 대표 행정동으로 삼습니다(population-weighted majority).
- 기간은 `seoul_link_speed_fact.csv`와 겹치는 2026-07-07~07-20(14일)만 사용합니다.

**시간 매칭 주의(14번 노트북과 다른 부분)**: 우리 `시간코드`는 1~24이고 **시간코드 i = 00:00~00:59 기준 i번째 버킷**이라 실제 시각은 `시간코드 - 1`시입니다(`pipeline/02_aggregate_5min_speed.py`의 `CREATHM 시 + 1` 규칙). 생활인구 원본의 `시간` 컬럼은 표준 0~23시 그대로입니다 — 직접 확인해보니 14번 노트북은 `시간코드`를 그대로 `시간`에 매칭하고 있어(즉 `-1` 보정 없음) 매 시간이 한 시간씩 밀려 매칭됩니다. 여기서는 `hour = 시간코드 - 1`로 보정해서 매칭합니다.

In [5]:
# 링크 -> 250m 격자(CELL_ID) 매칭 (링크 중심점이 속하는 셀)
link_pt = gpd.read_file(SEOUL_LINK_SHP)[["LINK_ID", "geometry"]]
link_pt["LINK_ID"] = link_pt["LINK_ID"].astype(str)
link_pt["geometry"] = link_pt.geometry.centroid
link_pt = link_pt.to_crs("EPSG:5179")

grid_cells = gpd.read_file(GRID_MATCH_SHP)[["CELL_ID", "geometry"]]
link_cell = gpd.sjoin(link_pt, grid_cells, how="left", predicate="within")[["LINK_ID", "CELL_ID"]]

print(f"링크->격자 매칭: {link_cell['CELL_ID'].notna().sum():,} / {len(link_cell):,}")

링크->격자 매칭: 63,506 / 63,548


In [6]:
# fact와 겹치는 14일치 격자/행정동 생활인구 로드 (시간코드 -> 0~23시 보정: hour = 시간코드 - 1)
pop_dates = sorted(df["기준일"].unique().astype(str))

grid_parts = []
for d in pop_dates:
    f = GRID_POP_DIR / f"250_LOCAL_RESD_{d}.csv"
    part = pd.read_csv(f, encoding="cp949", usecols=["일자", "시간", "행정동코드", "250M격자", "생활인구합계"])
    part["생활인구합계"] = pd.to_numeric(part["생활인구합계"], errors="coerce").fillna(0)
    grid_parts.append(part)
grid_raw = pd.concat(grid_parts, ignore_index=True)
print(f"grid_raw: {grid_raw.shape}")

dong_parts = []
for d in pop_dates:
    f = DONG_POP_DIR / f"250_LOCAL_RESD_ADMDONG_{d}.csv"
    part = pd.read_csv(f, encoding="cp949", usecols=["일자", "시간", "행정동코드", "생활인구합계"])
    part["생활인구합계"] = pd.to_numeric(part["생활인구합계"], errors="coerce").fillna(0)
    dong_parts.append(part)
dong_pop_lookup = pd.concat(dong_parts, ignore_index=True)
dong_pop_lookup.columns = ["기준일", "hour", "행정동코드", "dong_pop"]
dong_pop_lookup["기준일"] = dong_pop_lookup["기준일"].astype(str)
print(f"dong_pop_lookup: {dong_pop_lookup.shape}")

# 셀 -> 대표 행정동(인구 비중 최대) — 경계에 걸친 셀은 여러 행정동에 인구가 나뉘어 기록됨
cell_dong_weight = grid_raw.groupby(["250M격자", "행정동코드"])["생활인구합계"].sum().reset_index()
cell_to_dong = cell_dong_weight.loc[cell_dong_weight.groupby("250M격자")["생활인구합계"].idxmax()].set_index("250M격자")["행정동코드"]

# 셀 단위 생활인구(격자,일자,시간) — 경계 분할 행은 합산해서 셀 전체 인구로 복원
grid_pop_lookup = grid_raw.groupby(["250M격자", "일자", "시간"])["생활인구합계"].sum().reset_index()
grid_pop_lookup.columns = ["CELL_ID", "기준일", "hour", "grid_pop"]
grid_pop_lookup["기준일"] = grid_pop_lookup["기준일"].astype(str)
print(f"grid_pop_lookup: {grid_pop_lookup.shape}")

link_cell["행정동코드"] = link_cell["CELL_ID"].map(cell_to_dong)
print(f"링크->행정동(격자 경유) 매칭: {link_cell['행정동코드'].notna().sum():,} / {len(link_cell):,}")

dong_pop_lookup["행정동코드"] = dong_pop_lookup["행정동코드"].astype("float64")
link_cell_m = link_cell.copy()
link_cell_m["행정동코드"] = link_cell_m["행정동코드"].astype("float64")

df["hour"] = (df["시간코드"].astype(int) - 1)
df = df.merge(link_cell_m, on="LINK_ID", how="left")
df = df.merge(grid_pop_lookup, on=["CELL_ID", "기준일", "hour"], how="left")
df = df.merge(dong_pop_lookup, on=["행정동코드", "기준일", "hour"], how="left")
df["grid_pop"] = df["grid_pop"].astype("float32")
df["dong_pop"] = df["dong_pop"].astype("float32")

print(f"grid_pop 결측 없는 행 비율: {df['grid_pop'].notna().mean():.4f}")
print(f"dong_pop 결측 없는 행 비율: {df['dong_pop'].notna().mean():.4f}")
df[["grid_pop", "dong_pop"]].describe()

grid_raw: (3551636, 5)


dong_pop_lookup: (143472, 4)


grid_pop_lookup: (2866871, 4)
링크->행정동(격자 경유) 매칭: 63,318 / 63,548


grid_pop 결측 없는 행 비율: 0.9951
dong_pop 결측 없는 행 비율: 0.9958


,grid_pop,dong_pop
count,1.502026e+07,1.503117e+07
mean,1.775540e+03,3.164569e+04
std,1.826875e+03,2.011730e+04
min,0.000000e+00,2.489320e+03
25%,3.869800e+02,2.023400e+04
50%,1.429770e+03,2.753769e+04
75%,2.583510e+03,3.774350e+04
max,5.454269e+04,2.274385e+05


## 5. (신규, 9차) View-T 실측 평균속도·혼잡지표

`AverageSpeed(LINK).xlsx`(2024년 평일 링크별 실측 평균속도, 연간+저녁첨두+시간대별)와 `CongestIndex(LINK).xlsx`(혼잡빈도·시간·기대강도)를 추가합니다. `traffic_volume.xlsx`와 완전히 같은 매칭 방식(Level5.5 LINK ID → `ITS LINK ID` 콤마 explode)입니다.

**사전 확인(별도 검증 완료)**: 두 파일 다 `traffic_volume_aadt`가 커버하는 것과 **정확히 같은 17,929개 링크**(Level5.5 주요도로망)만 커버합니다 — C 계층(이웃없음) 커버리지 1.1%로 새 도로를 못 덮습니다. 다만 `annual_avg_speed`는 우리 실측 링크평균속도와 상관계수 **0.787**(지금까지 중 최강), `MAX_SPD`와의 상관은 0.460뿐이라 완전 중복은 아닙니다. 시간대별 `avg_speed_h*`는 4절에서 만든 `hour`(=시간코드-1, 0~23시)로 매칭합니다.

In [7]:
def explode_its(frame):
    frame = frame.copy()
    frame["its_id_list"] = frame["ITS_LINK_ID"].astype(str).str.split(",")
    exp = frame.explode("its_id_list")
    exp["LINK_ID"] = exp["its_id_list"].str.strip()
    return exp[exp["LINK_ID"] != "0"]


AS_COLS = (["LEVEL55_LINK_ID", "ITS_LINK_ID", "road_rank_as", "road_name_as", "region_as",
            "length_km_as", "lanes_as", "annual_avg_speed", "evening_peak_speed"]
           + [f"speed_h{h}" for h in range(24)])
avg_speed = pd.read_excel(AVG_SPEED_XLSX, sheet_name="Sheet0", header=None, skiprows=7).iloc[:, :33]
avg_speed.columns = AS_COLS
# 원본에 숫자/문자가 섞여 object(또는 pandas string) dtype으로 읽히는 경우가 있어 명시적으로 숫자 변환
AS_NUMERIC_COLS = ["annual_avg_speed", "evening_peak_speed"] + [f"speed_h{h}" for h in range(24)]
for col in AS_NUMERIC_COLS:
    avg_speed[col] = pd.to_numeric(avg_speed[col], errors="coerce")
as_exp = explode_its(avg_speed)

avg_speed_lookup = as_exp.groupby("LINK_ID", as_index=False)[["annual_avg_speed", "evening_peak_speed"]].mean()

hourly_cols = [f"speed_h{h}" for h in range(24)]
as_long = as_exp.melt(id_vars="LINK_ID", value_vars=hourly_cols, var_name="hcol", value_name="avg_speed_hourly")
as_long["hour"] = as_long["hcol"].str.replace("speed_h", "", regex=False).astype(int)
as_long = as_long.groupby(["LINK_ID", "hour"], as_index=False)["avg_speed_hourly"].mean()

CI_COLS = ["LEVEL55_LINK_ID", "ITS_LINK_ID", "road_rank_ci", "road_name_ci", "region_ci",
           "length_km_ci", "lanes_ci", "congest_freq", "congest_time", "congest_expect"]
congest = pd.read_excel(CONGEST_INDEX_XLSX, sheet_name="Sheet0", header=6, skiprows=[7])
congest.columns = CI_COLS
CI_NUMERIC_COLS = ["congest_freq", "congest_time", "congest_expect"]
for col in CI_NUMERIC_COLS:
    congest[col] = pd.to_numeric(congest[col], errors="coerce")
ci_exp = explode_its(congest)
congest_lookup = ci_exp.groupby("LINK_ID", as_index=False)[["congest_freq", "congest_time", "congest_expect"]].mean()

print(f"AverageSpeed 링크 매칭: {avg_speed_lookup['LINK_ID'].isin(df['LINK_ID']).sum():,} / {avg_speed_lookup.shape[0]:,}")
print(f"CongestIndex 링크 매칭: {congest_lookup['LINK_ID'].isin(df['LINK_ID']).sum():,} / {congest_lookup.shape[0]:,}")

df = df.merge(avg_speed_lookup, on="LINK_ID", how="left")
df = df.merge(as_long, on=["LINK_ID", "hour"], how="left")
df = df.merge(congest_lookup, on="LINK_ID", how="left")

VIEWT_COLS = ["annual_avg_speed", "evening_peak_speed", "avg_speed_hourly", "congest_freq", "congest_time", "congest_expect"]
for col in VIEWT_COLS:
    df[col] = df[col].astype("float32")

print(f"행 기준 매칭률: {df['annual_avg_speed'].notna().mean():.1%}")
df[VIEWT_COLS].describe()

C:\Users\ejjun\AppData\Roaming\Python\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


C:\Users\ejjun\AppData\Roaming\Python\Python313\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


AverageSpeed 링크 매칭: 17,675 / 18,164


CongestIndex 링크 매칭: 17,163 / 17,642


행 기준 매칭률: 39.3%


,annual_avg_speed,evening_peak_speed,avg_speed_hourly,congest_freq,congest_time,congest_expect
count,5.938480e+06,5.934112e+06,5.935218e+06,5.766448e+06,5.766448e+06,5766448.0
mean,2.989737e+01,2.753609e+01,3.166991e+01,5.169582e+01,6.928394e+01,0.0
std,9.902841e+00,9.825967e+00,1.167102e+01,2.492275e+01,2.052282e+01,0.0
min,5.000000e+00,5.000000e+00,3.000000e+00,0.000000e+00,0.000000e+00,0.0
25%,2.300000e+01,2.100000e+01,2.400000e+01,3.400000e+01,5.700000e+01,0.0
50%,2.800000e+01,2.566670e+01,3.000000e+01,4.800000e+01,7.050000e+01,0.0
75%,3.400000e+01,3.200000e+01,3.700000e+01,6.900000e+01,8.500000e+01,0.0
max,9.600000e+01,9.666670e+01,1.500000e+02,1.000000e+02,1.000000e+02,0.0


## 6. 피처 세트 정의 (8차 vs 9차)

이제부터 GroupKFold 한 번으로 두 구성을 같이 비교합니다.
- **8차**: 도로기본/지형/시간 + 외부컨텍스트 + 이웃속도 + 도로망 위상(7차) + 강수·첨두파생(7차) + 생활인구(6차)
- **9차**: 8차 전체 + View-T `annual_avg_speed`/`evening_peak_speed`/`avg_speed_hourly`/`congest_freq`/`congest_time`/`congest_expect`

두 구성 모두 B/C 분리모델(이웃속도 포함 모델 + 정적 전용 모델)로 평가합니다.

In [8]:
SPARSE_NUMERIC_COLS = SPARSE_NUMERIC_COLS + ["nearest_obs_neighbor_speed"]
POPULATION_COLS = ["grid_pop", "dong_pop"]
RAIN_PEAK_NUMERIC_COLS = [
    "is_rain", "log_precip", "is_morning_peak", "is_evening_peak",
    "rain_morning_peak", "rain_evening_peak", "rain_x_max_spd",
]
RAIN_PEAK_CATEGORICAL_COLS = ["rain_class"]
DERIVED_COLS = (
    ["dow", "is_weekend", "hour_sin", "hour_cos"]
    + EXTERNAL_CONTEXT_COLS
    + SPARSE_NUMERIC_COLS
)

# 8차: 7차까지의 전체 피처(도로기본/지형/시간 + 외부컨텍스트 + 이웃속도 + 위상 + 강수·첨두파생 + 생활인구)
FULL_FEATURE_COLS = (
    NUMERIC_COLS
    + CATEGORICAL_COLS
    + DERIVED_COLS
    + TOPO_COLS
    + POPULATION_COLS
    + RAIN_PEAK_NUMERIC_COLS
    + RAIN_PEAK_CATEGORICAL_COLS
)
FULL_CATEGORICAL_COLS = CATEGORICAL_COLS + RAIN_PEAK_CATEGORICAL_COLS

# 9차: 8차 + View-T 실측 평균속도·혼잡지표
VIEWT_FEATURE_COLS = FULL_FEATURE_COLS + VIEWT_COLS

print("8차(FULL_FEATURE_COLS) 피처 수:", len(FULL_FEATURE_COLS))
print("9차(VIEWT_FEATURE_COLS) 피처 수:", len(VIEWT_FEATURE_COLS))
df[DERIVED_COLS + POPULATION_COLS + RAIN_PEAK_NUMERIC_COLS + RAIN_PEAK_CATEGORICAL_COLS].describe(include="all")

8차(FULL_FEATURE_COLS) 피처 수: 72
9차(VIEWT_FEATURE_COLS) 피처 수: 78


,dow,is_weekend,hour_sin,hour_cos,bus_lane_ratio,school_zone_count,accident_hotspot_count,speed_bump_count,traffic_volume_aadt,traffic_volume_hourly,...,grid_pop,dong_pop,is_rain,log_precip,is_morning_peak,is_evening_peak,rain_morning_peak,rain_evening_peak,rain_x_max_spd,rain_class
count,15094335.0,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,5.938480e+06,5.938480e+06,...,1.502026e+07,1.503117e+07,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,1.509434e+07,15094335
unique,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5
top,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,none
freq,2157217.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11536235
mean,NaN,2.857848e-01,-2.138451e-05,7.420300e-06,6.608189e-02,6.103236e-01,3.469555e-01,2.866067e-01,2.066972e+04,8.612402e+02,...,1.775540e+03,3.164569e+04,2.357242e-01,6.981273e-01,1.249963e-01,1.250046e-01,3.565914e-02,1.768922e-02,2.934275e+01,NaN
std,NaN,4.517874e-01,7.071049e-01,7.071076e-01,2.248980e-01,8.294121e-01,1.594318e+00,8.003197e-01,1.513154e+04,7.628351e+02,...,1.826875e+03,2.011730e+04,4.244506e-01,1.213060e+00,3.307147e-01,3.307242e-01,1.854389e-01,1.318192e-01,5.454292e+01,NaN
min,NaN,0.000000e+00,-1.000000e+00,-1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.093000e+03,0.000000e+00,...,0.000000e+00,2.489320e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN
25%,NaN,0.000000e+00,-7.071067e-01,-7.071068e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.115400e+04,3.570000e+02,...,3.869800e+02,2.023400e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN
50%,NaN,0.000000e+00,-8.742278e-08,1.192488e-08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.748900e+04,6.600000e+02,...,1.429770e+03,2.753769e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN
75%,NaN,1.000000e+00,7.071068e-01,7.071070e-01,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,2.567400e+04,1.144000e+03,...,2.583510e+03,3.774350e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN


## 7. 공정한 평가 — Section/Link GroupKFold (공간 일반화, 8차 vs 9차)

Model B가 진짜 겨루는 문제는 "한 번도 안 본 도로"를 맞히는 **공간 일반화**(section GroupKFold, T-data 기준 **R²=0.641**)입니다. 5분 데이터는 T-data 때와 달리 관측 복제가 없어서 `source_section_id = LINK_ID`(`src/pipeline/05_build_speed_fact.py` 근거) — 즉 지금은 **LINK_ID 자체가 곧 section**입니다. `LINK_ID` 기준 GroupKFold를 쓰면 폴드마다 특정 링크들이 통째로 test로 빠져서, 그 링크의 어떤 시간대 행도 학습에 들어가지 않습니다 — 이게 **13번 노트북의 B/C 계층과 정확히 같은 원리**입니다(그 폴드의 train 링크가 곧 "관측", test 링크가 곧 "미관측").

**중요한 예외 처리**: `nearest_obs_neighbor_speed`는 우리 자신의 관측 데이터에서 파생된 피처라, 폴드마다 다시 계산해야 합니다 — "이 폴드에서 미관측이라고 치는 링크"의 실측값이 다른 링크의 이웃 피처로 새어 들어가면(=train에 있는 링크가 test 링크의 실측값을 이웃 피처로 참조) 평가가 부풀려집니다. 인접 그래프(`link_neighbors`)와 링크x시간 실측 행렬(`filled`/`valid`)은 3번 섹션에서 만든 걸 재사용하되, **그 폴드의 train 링크만 "이웃으로 인정"**하도록 희소행렬만 다시 만듭니다. 도로속성/위상/외부컨텍스트/생활인구/강수파생/View-T는 전부 외부 데이터이거나 우리 라벨과 무관한 정적 계산값이라 이 문제가 없어 그대로 재사용합니다.

**8차와 9차를 같은 루프에서 같이 평가합니다** — 피처 구성(`FULL_FEATURE_COLS` / `VIEWT_FEATURE_COLS`)마다, 폴드마다 이웃속도 포함 모델과 정적 전용 모델을 둘 다 학습하고, 그 폴드의 test 링크를 "이번 폴드 기준 이웃 있음(B)/없음(C)"으로 나눠 각각 다른 모델로 예측·평가합니다(B/C 분리모델). 참고용으로 이웃속도 모델 단독(분리 전, "단일모델") 예측도 같이 남겨 분리모델이 실제로 C 계층을 얼마나 회복시키는지 대조합니다. 구성 2개 x 폴드 3개 x 모델 2개 = 총 12번의 LightGBM 학습이라 시간이 꽤 걸립니다.

3-fold로 진행합니다(팀 `experiments/eval_protocols.py`도 3-fold 기본값을 씁니다).

In [9]:
from sklearn.model_selection import GroupKFold

df["tm_key"] = df["기준일"].astype(str) + "_" + df["시간코드"].astype(int).astype(str)

N_FOLDS = 3
groups = df["LINK_ID"].values
gkf = GroupKFold(n_splits=N_FOLDS)

CONFIGS = [
    ("8차", FULL_FEATURE_COLS),
    ("9차", VIEWT_FEATURE_COLS),
]


def build_neighbor_feature_for_allowed(allowed_links):
    """allowed_links(그 폴드의 train 링크)만 이웃으로 인정해 인접 실측 속도 재계산."""
    rows_i, cols_i = [], []
    for l in link_ids_sorted:
        i = link_idx[l]
        for n in link_neighbors.get(l, ()):
            if n in allowed_links:
                rows_i.append(i)
                cols_i.append(link_idx[n])
    A_fold = csr_matrix((np.ones(len(rows_i), dtype="float32"), (rows_i, cols_i)), shape=(n_links, n_links))
    num = A_fold @ filled
    den = A_fold @ valid
    with np.errstate(invalid="ignore", divide="ignore"):
        nm = num / den
    nm[den == 0] = np.nan
    return pd.DataFrame(nm, index=link_ids_sorted, columns=tm_columns) \
        .reset_index().rename(columns={"index": "LINK_ID"}) \
        .melt(id_vars="LINK_ID", var_name="tm_key", value_name="fold_neighbor_speed")


def evaluate(y_true, pred, name):
    return {"model": name,
            "rmse": mean_squared_error(y_true, pred) ** 0.5,
            "mae": mean_absolute_error(y_true, pred),
            "r2": r2_score(y_true, pred)}


def eval_bucket(y_true, pred, mask, name):
    if mask.sum() < 30:
        return {"model": name, "n": int(mask.sum()), "rmse": float("nan"), "mae": float("nan"), "r2": float("nan")}
    return {"model": name, "n": int(mask.sum()),
            "rmse": mean_squared_error(y_true[mask], pred[mask]) ** 0.5,
            "mae": mean_absolute_error(y_true[mask], pred[mask]),
            "r2": r2_score(y_true[mask], pred[mask])}


fold_rows = []
fold_tier_rows = []
last_models = {}  # config -> 마지막 폴드의 이웃속도 모델(피처중요도용)

for config_name, feature_cols in CONFIGS:
    static_cols = [c for c in feature_cols if c != "nearest_obs_neighbor_speed"]

    for fold_i, (train_idx, test_idx) in enumerate(gkf.split(df, groups=groups)):
        train_links = set(df["LINK_ID"].iloc[train_idx].unique())
        test_links = set(df["LINK_ID"].iloc[test_idx].unique())
        assert len(train_links & test_links) == 0, "링크가 train/test에 겹치면 안 됨"

        nm_long = build_neighbor_feature_for_allowed(train_links)
        fold_neighbor = df[["LINK_ID", "tm_key"]].merge(nm_long, on=["LINK_ID", "tm_key"], how="left")["fold_neighbor_speed"]

        Xf = df[feature_cols].copy()
        Xf["nearest_obs_neighbor_speed"] = fold_neighbor.to_numpy(dtype="float32")
        yf = df[TARGET]

        X_train, X_test = Xf.iloc[train_idx], Xf.iloc[test_idx]
        y_train, y_test = yf.iloc[train_idx], yf.iloc[test_idx]

        model_nb = lgb.LGBMRegressor(objective="regression", n_estimators=300, learning_rate=0.05,
                                     num_leaves=63, random_state=RANDOM_STATE, verbosity=-1)
        model_nb.fit(X_train, y_train, categorical_feature=FULL_CATEGORICAL_COLS + ["dow"])
        pred_nb = model_nb.predict(X_test)

        model_stat = lgb.LGBMRegressor(objective="regression", n_estimators=300, learning_rate=0.05,
                                       num_leaves=63, random_state=RANDOM_STATE, verbosity=-1)
        model_stat.fit(X_train[static_cols], y_train, categorical_feature=FULL_CATEGORICAL_COLS + ["dow"])
        pred_stat = model_stat.predict(X_test[static_cols])

        has_nb_test = X_test["nearest_obs_neighbor_speed"].notna().to_numpy()
        pred_split = np.where(has_nb_test, pred_nb, pred_stat)
        y_test_arr = y_test.to_numpy()

        row_single = evaluate(y_test_arr, pred_nb, f"{config_name}_fold{fold_i}_단일모델")
        row_split = evaluate(y_test_arr, pred_split, f"{config_name}_fold{fold_i}_분리모델")
        for row, variant in ((row_single, "단일모델"), (row_split, "분리모델")):
            row["config"] = config_name
            row["variant"] = variant
            row["fold"] = fold_i
            row["n_train_links"] = len(train_links)
            row["n_test_links"] = len(test_links)
        fold_rows.append(row_single)
        fold_rows.append(row_split)

        fold_tier_rows.append({**eval_bucket(y_test_arr, pred_nb, has_nb_test, f"{config_name}_단일모델_B"),
                               "config": config_name, "variant": "단일모델", "fold": fold_i, "tier": "B"})
        fold_tier_rows.append({**eval_bucket(y_test_arr, pred_nb, ~has_nb_test, f"{config_name}_단일모델_C"),
                               "config": config_name, "variant": "단일모델", "fold": fold_i, "tier": "C"})
        fold_tier_rows.append({**eval_bucket(y_test_arr, pred_split, has_nb_test, f"{config_name}_분리모델_B"),
                               "config": config_name, "variant": "분리모델", "fold": fold_i, "tier": "B"})
        fold_tier_rows.append({**eval_bucket(y_test_arr, pred_split, ~has_nb_test, f"{config_name}_분리모델_C"),
                               "config": config_name, "variant": "분리모델", "fold": fold_i, "tier": "C"})

        print(f"[{config_name}] fold{fold_i}: train_links={len(train_links):,} test_links={len(test_links):,} "
              f"이웃있음(B) 비율={has_nb_test.mean():.1%}")
        print(f"  단일모델 R2={row_single['r2']:.3f} MAE={row_single['mae']:.3f}"
              f" | 분리모델 R2={row_split['r2']:.3f} MAE={row_split['mae']:.3f}")

        last_models[config_name] = model_nb

        del nm_long, fold_neighbor, Xf, X_train, X_test, y_train, y_test, y_test_arr, pred_nb, pred_stat, pred_split
        gc.collect()

group_results = pd.DataFrame(fold_rows)
group_tier_results = pd.DataFrame(fold_tier_rows)
group_results

[8차] fold0: train_links=29,973 test_links=14,987 이웃있음(B) 비율=94.2%
  단일모델 R2=0.708 MAE=4.589 | 분리모델 R2=0.708 MAE=4.586


[8차] fold1: train_links=29,974 test_links=14,986 이웃있음(B) 비율=94.5%
  단일모델 R2=0.701 MAE=4.632 | 분리모델 R2=0.703 MAE=4.626


[8차] fold2: train_links=29,973 test_links=14,987 이웃있음(B) 비율=94.4%
  단일모델 R2=0.715 MAE=4.608 | 분리모델 R2=0.716 MAE=4.608


[9차] fold0: train_links=29,973 test_links=14,987 이웃있음(B) 비율=94.2%
  단일모델 R2=0.727 MAE=4.439 | 분리모델 R2=0.727 MAE=4.435


[9차] fold1: train_links=29,974 test_links=14,986 이웃있음(B) 비율=94.5%
  단일모델 R2=0.721 MAE=4.465 | 분리모델 R2=0.722 MAE=4.462


[9차] fold2: train_links=29,973 test_links=14,987 이웃있음(B) 비율=94.4%
  단일모델 R2=0.734 MAE=4.444 | 분리모델 R2=0.735 MAE=4.442


,model,rmse,mae,r2,config,variant,fold,n_train_links,n_test_links
0,8차_fold0_단일모델,6.601165,4.589142,0.707757,8차,단일모델,0,29973,14987
1,8차_fold0_분리모델,6.593248,4.585654,0.708458,8차,분리모델,0,29973,14987
2,8차_fold1_단일모델,6.705151,4.632297,0.701089,8차,단일모델,1,29974,14986
3,8차_fold1_분리모델,6.687936,4.626392,0.702622,8차,분리모델,1,29974,14986
4,8차_fold2_단일모델,6.550575,4.608493,0.715413,8차,단일모델,2,29973,14987
5,8차_fold2_분리모델,6.546854,4.608110,0.715736,8차,분리모델,2,29973,14987
6,9차_fold0_단일모델,6.384854,4.438510,0.726596,9차,단일모델,0,29973,14987
7,9차_fold0_분리모델,6.383676,4.435054,0.726697,9차,분리모델,0,29973,14987
8,9차_fold1_단일모델,6.472659,4.465312,0.721458,9차,단일모델,1,29974,14986
9,9차_fold1_분리모델,6.462855,4.461804,0.722301,9차,분리모델,1,29974,14986


In [10]:
# 계층(B/C)별 3-fold 평균 — 이게 지난 대화에서 물었던 "C 계층이 진짜 병목인가"에 대한 정직한 답
tier_summary = group_tier_results.groupby(["config", "variant", "tier"])[["n", "rmse", "mae", "r2"]].mean()
print("=== 계층별 3-fold 평균 (공간 일반화, 8차 vs 9차) ===")
print(tier_summary.round(3).to_string())
print()
for config_name in ["8차", "9차"]:
    b_single = tier_summary.loc[(config_name, "단일모델", "B")]
    c_single = tier_summary.loc[(config_name, "단일모델", "C")]
    b_split = tier_summary.loc[(config_name, "분리모델", "B")]
    c_split = tier_summary.loc[(config_name, "분리모델", "C")]
    print(f"[{config_name}] C 계층(이웃없음) 단일모델 R2={c_single['r2']:.3f} -> 분리모델 R2={c_split['r2']:.3f} "
          f"(Δ{c_split['r2']-c_single['r2']:+.3f})")
    print(f"[{config_name}] B 계층(이웃있음) 단일모델 R2={b_single['r2']:.3f} -> 분리모델 R2={b_split['r2']:.3f} "
          f"(Δ{b_split['r2']-b_single['r2']:+.3f}, 이론상 동일해야 하며 차이는 순전히 랜덤성)")


=== 계층별 3-fold 평균 (공간 일반화, 8차 vs 9차) ===
                               n   rmse    mae     r2
config variant tier                                  
8차     단일모델    B     4748189.667  6.509  4.518  0.722
               C      283255.333  8.241  6.157  0.324
       분리모델    B     4748189.667  6.509  4.518  0.722
               C      283255.333  8.102  6.099  0.346
9차     단일모델    B     4748189.667  6.288  4.357  0.741
               C      283255.333  7.985  6.000  0.364
       분리모델    B     4748189.667  6.288  4.357  0.741
               C      283255.333  7.887  5.942  0.380

[8차] C 계층(이웃없음) 단일모델 R2=0.324 -> 분리모델 R2=0.346 (Δ+0.021)
[8차] B 계층(이웃있음) 단일모델 R2=0.722 -> 분리모델 R2=0.722 (Δ+0.000, 이론상 동일해야 하며 차이는 순전히 랜덤성)
[9차] C 계층(이웃없음) 단일모델 R2=0.364 -> 분리모델 R2=0.380 (Δ+0.016)
[9차] B 계층(이웃있음) 단일모델 R2=0.741 -> 분리모델 R2=0.741 (Δ+0.000, 이론상 동일해야 하며 차이는 순전히 랜덤성)


In [11]:
split_fold_results = group_results[group_results["variant"] == "분리모델"]
group_summary_by_config = split_fold_results.groupby("config")[["rmse", "mae", "r2"]].mean()

print("=== 3-fold 평균 (공간 일반화, B/C 분리모델, 정직한 비교) ===")
print(group_summary_by_config.round(3).to_string())
print()
print("=== 벤치마크 비교표 ===")
compare = pd.DataFrame([
    {"model": "Model B 공간일반화 (T-data, 팀 공식 벤치마크)", "rmse": 7.71, "mae": 5.61, "r2": 0.641},
    {"model": "Model B 공간일반화 (5분 데이터, src/11 재실행, 우리 피처 없음)", "rmse": 8.104321, "mae": 5.947932, "r2": 0.562342},
    {"model": "Model B 시간일반화 (T-data, 참고용)", "rmse": 6.43, "mae": 4.72, "r2": 0.740},
    {"model": "8차(위상+강수+인구, B/C 분리모델)", **group_summary_by_config.loc["8차"].to_dict()},
    {"model": "9차(8차 + ViewT 평균속도/혼잡, B/C 분리모델)", **group_summary_by_config.loc["9차"].to_dict()},
])
print(compare[["model", "rmse", "mae", "r2"]].to_string(index=False))

=== 3-fold 평균 (공간 일반화, B/C 분리모델, 정직한 비교) ===
         rmse    mae     r2
config                     
8차      6.609  4.607  0.709
9차      6.389  4.446  0.728

=== 벤치마크 비교표 ===
                                       model     rmse      mae       r2
           Model B 공간일반화 (T-data, 팀 공식 벤치마크) 7.710000 5.610000 0.641000
Model B 공간일반화 (5분 데이터, src/11 재실행, 우리 피처 없음) 8.104321 5.947932 0.562342
                 Model B 시간일반화 (T-data, 참고용) 6.430000 4.720000 0.740000
                      8차(위상+강수+인구, B/C 분리모델) 6.609346 4.606719 0.708939
            9차(8차 + ViewT 평균속도/혼잡, B/C 분리모델) 6.388552 4.446168 0.728055


## 8. 피처 중요도 (9차, GroupKFold 마지막 폴드 이웃속도모델 기준)

무작위 분할이 아니라 방금 GroupKFold로 실제 학습된 모델(`last_models["9차"]`, fold2의 이웃속도 포함 모델)에서 gain 기준 중요도를 봅니다 — 공간 일반화 문제에서 실제로 쓰인 모델 기준이라 이전(무작위 분할) 버전보다 더 정직한 순위입니다.

In [12]:
model_9cha = last_models["9차"]
importance = pd.DataFrame({
    "feature": model_9cha.feature_name_,
    "gain": model_9cha.booster_.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)
importance["gain_share"] = importance["gain"] / importance["gain"].sum()

NEW_COLS_ALL = DERIVED_COLS + POPULATION_COLS + TOPO_COLS + RAIN_PEAK_NUMERIC_COLS + RAIN_PEAK_CATEGORICAL_COLS + VIEWT_COLS
importance["group"] = np.select(
    [importance["feature"] == "nearest_obs_neighbor_speed",
     importance["feature"].isin(VIEWT_COLS),
     importance["feature"].isin(TOPO_COLS),
     importance["feature"].isin(RAIN_PEAK_NUMERIC_COLS + RAIN_PEAK_CATEGORICAL_COLS),
     importance["feature"].isin(POPULATION_COLS),
     importance["feature"].isin(EXTERNAL_CONTEXT_COLS + SPARSE_NUMERIC_COLS)],
    ["이웃속도", "ViewT(9차)", "위상·파생(7차)", "강수·첨두파생(7차)", "생활인구(6차)", "외부컨텍스트"],
    default="도로기본/지형/시간",
)
importance["is_new"] = importance["feature"].isin(NEW_COLS_ALL)

print("=== 신규/파생 피처 순위(9차, 이웃속도 포함 모델) ===")
print(importance[importance["is_new"]].head(25)[["feature", "group", "gain_share"]].to_string(index=False))
print()
print("=== 그룹별 gain 합계 ===")
print(importance.groupby("group")["gain_share"].sum().sort_values(ascending=False).round(4).to_string())
print()
print("=== 전체 상위 15개 ===")
print(importance.head(15)[["feature", "group", "gain_share"]].round(4).to_string(index=False))

=== 신규/파생 피처 순위(9차, 이웃속도 포함 모델) ===
                     feature     group  gain_share
  nearest_obs_neighbor_speed      이웃속도    0.640083
            avg_speed_hourly ViewT(9차)    0.086252
t_node_incoming_max_spd_mean 위상·파생(7차)    0.017148
             max_spd_x_lanes 위상·파생(7차)    0.016306
            annual_avg_speed ViewT(9차)    0.013840
    t_node_node_total_degree 위상·파생(7차)    0.008626
    length_x_node_complexity 위상·파생(7차)    0.007698
      downstream_speed_delta 위상·파생(7차)    0.007672
                    grid_pop  생활인구(6차)    0.004995
      f_node_node_out_degree 위상·파생(7차)    0.004794
  t_node_incoming_lanes_mean 위상·파생(7차)    0.004750
         traffic_volume_aadt    외부컨텍스트    0.004349
          evening_peak_speed ViewT(9차)    0.003961
    f_node_node_total_degree 위상·파생(7차)    0.003778
             node_complexity 위상·파생(7차)    0.003619
  f_node_outgoing_lanes_mean 위상·파생(7차)    0.002859
f_node_incoming_max_spd_mean 위상·파생(7차)    0.002846
                congest_time ViewT(9차)    0.00

## 9. 해석 및 한계

**해석 기준**: 8차/9차 모두 `group_summary_by_config`(공간 일반화, B/C 분리모델)를 Model B 공간일반화 두 벤치마크(T-data 0.641, 5분 데이터+팀 피처만 0.562342)와 비교하는 게 정직한 비교입니다.

**8차 대비 9차 효과**: `annual_avg_speed` 등 View-T 피처는 `traffic_volume_aadt`와 정확히 같은 17,929개 링크(Level5.5)만 커버하고 C 계층 커버리지는 1.1%에 불과합니다 — 위 계층별 표(7절)에서 9차의 C 계층 R²가 8차 대비 거의 안 움직였다면 이 커버리지 한계 때문이고, 반대로 B 계층이나 전체 R²가 눈에 띄게 올랐다면 커버되는 링크에서 `annual_avg_speed`(상관계수 0.787)의 신호가 실제로 쓰였다는 뜻입니다. 진짜 C 계층 병목 해소는 Level6 상세도로망 데이터(공문 필요, 아직 미확보)가 있어야 합니다.

**계층별 결과(7절)가 핵심**입니다 — B/C 분리모델을 쓰면 C 계층(이웃없음)의 R²가 단일모델 대비 얼마나 회복되는지가, "이웃속도 피처를 그대로 쓰면 정작 정보가 가장 부족한 링크에서 해를 끼친다"는 13번 노트북의 발견을 이 노트북의 전체 피처 집합(위상+생활인구+강수파생+View-T 포함)에서도 재현하는지 보여줍니다. B 계층은 두 방식이 이론상 동일해야 하므로(둘 다 이웃속도 포함 모델을 그대로 씀), 차이가 있다면 순전히 랜덤성입니다.

`models/11`의 팀 피처(노드 위상 등, `pipeline/04` 산출)와 이 노트북의 피처는 이제 도로망 위상까지 포함해 거의 같은 집합이 됐지만(생활인구·View-T는 이 노트북에만 있음), `models/11`은 아직 새 스키마로 옮겨지지 않았습니다(입력 경로가 삭제된 옛 병합 파일을 가리키는 상태) — 완전한 최종 비교는 `models/11`을 새 스키마로 옮긴 뒤 나옵니다.

**남은 가정/한계**
- 추정교통량(`traffic_volume_hourly`/`_aadt`)의 시간대 매칭(시간코드 1=0~1시)은 공식 문서로 확인된 게 아니라 순서 기반 가정이고, 2024년 평일 기준이라 2026년 7월(주말 포함) 관측과 시점·요일 구성이 다릅니다. View-T `AverageSpeed`/`CongestIndex`도 같은 2024년 평일 기준이라 동일한 한계를 공유합니다.
- 스쿨존 200m 버퍼는 직선거리 근사(도로 형상 기준 아님)이고, `school_zone_count`는 `MAX_SPD`와 일부 중복 정보일 수 있습니다. 과속방지턱 원본 도형은 닫히지 않은 링 오류가 있어 `on_invalid="fix"`로 보정해서 읽었습니다.
- 생활인구(6절): 격자/행정동 원본의 소량값 마스킹(`*`)을 0으로 처리했습니다(임시 처리). 링크→격자 매칭은 링크 중심점 1개로만 판정해 긴 링크의 실제 인구 노출과는 오차가 있을 수 있습니다. `grid_pop`/`dong_pop` 둘 다 대략 95% 매칭이라 나머지는 결측으로 남아 LightGBM 기본 결측 처리에 맡깁니다.
- 도로망 위상·파생 28개(7절)는 `notebooks/13_tier_split_evaluation.ipynb`가 **공간 블록분할** 기준으로 먼저 검증한 값입니다(B +0.026~+0.031, C +0.043) — 이 노트북의 GroupKFold 결과가 그 규모를 그대로 재현하지 않을 수 있습니다(피처 집합·폴드 구성이 다름).
- `routing/13`은 이미 B/C 분리모델 방식을 실전에 반영했습니다 — 이 노트북에서 View-T 효과가 확인되면 다음 단계로 `routing/13`에도 반영할 수 있습니다.